# 🚀 Huấn luyện IoT Device Identification & Backup lên Cloudflare R2

**LƯU Ý:** Vì bạn không dùng Google Drive, bạn cần đảm bảo file `dataset.zip` và các file model pretrain cũ đã được đẩy lên R2 Bucket của bạn trước, hoặc bạn có một đường link tải trực tiếp (wget).

In [ ]:
# 1. Cài đặt thư viện AWS SDK (boto3)
!pip install -q boto3 torch tqdm scikit-learn pandas numpy scipy pyyaml joblib matplotlib seaborn

In [ ]:
# 2. Cấu hình Cloudflare R2
import os
import boto3
from botocore.config import Config

R2_ACCOUNT_ID = "nhap_account_id_cua_ban"
R2_ACCESS_KEY = "nhap_access_key"
R2_SECRET_KEY = "nhap_secret_key"
BUCKET_NAME = "ten-bucket-cua-ban"

s3 = boto3.client(
    's3',
    endpoint_url=f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=R2_ACCESS_KEY,
    aws_secret_access_key=R2_SECRET_KEY,
    config=Config(signature_version='s3v4')
)

print("✅ Đã cấu hình xong Client kết nối R2!")

In [ ]:
# 3. Tải Data và Code
import os

# Kéo code từ GitHub
%cd /content
!rm -rf IoT
!git clone https://github.com/hnihTyoB/IoT.git
%cd /content/IoT

# Lấy dataset từ R2 (Giả sử bạn đã up file dataset.zip lên R2)
print("\n--- ĐANG TẢI DATASET TỪ R2 ---")
try:
    s3.download_file(BUCKET_NAME, "dataset.zip", "/content/dataset.zip")
    !unzip -q /content/dataset.zip -d /content/
    print("✅ Đã giải nén dataset!")
except Exception as e:
    print("⚠️ Lỗi tải dataset từ R2 (bạn có thể bỏ qua nếu đã tự wget):", e)

# Lấy kết quả experiments cũ (để lấy model Phase 1, 2) từ R2
print("\n--- ĐANG TẢI EXPERIMENTS CŨ TỪ R2 ---")
try:
    !mkdir -p /content/IoT/experiments
    s3.download_file(BUCKET_NAME, "experiments_backup.zip", "/content/experiments_backup.zip")
    !unzip -q /content/experiments_backup.zip -d /content/IoT/experiments/
    print("✅ Đã giải nén model cũ!")
except Exception as e:
    print("⚠️ Không tìm thấy file experiments_backup.zip trên R2. Bắt đầu train mới hoàn toàn.")

In [ ]:
# 4. Chạy quá trình huấn luyện
# Ở đây để mặc định chạy Phase 3 (Finetune), nếu muốn có thể thêm các lệnh chạy Phase 1,2.
print("\n--- ĐANG CHẠY PHASE 3: FINETUNE ROBUST ---")
!python run_experiments.py --config configs/finetune_unsw.yaml

In [ ]:
# 5. Chạy chương trình nhận diện
!python inference.py

In [ ]:
# 6. Đóng gói kết quả và đẩy lên R2
print("\n--- ĐÓNG GÓI KẾT QUẢ ---")
!zip -r /content/finetune_robust_frozen_v2.zip /content/IoT/experiments/finetune_robust_frozen/

print("\n--- ĐANG UPLOAD LÊN CLOUDFLARE R2 ---")
file_path = "/content/finetune_robust_frozen_v2.zip"
object_name = "finetune_robust_frozen_v2.zip"

try:
    s3.upload_file(file_path, BUCKET_NAME, object_name)
    print(f"✅ Tải lên thành công! File lưu tại bucket '{BUCKET_NAME}' với tên '{object_name}'")
except Exception as e:
    print("❌ Upload thất bại:", e)